# 02: Flow Method Comparison (PIV, 2D Neural, 3D Native)

Estimate the surface velocity field with all three methods on one frame pair and
compare their kinematics with the A1-style dashboard (`plot_flow_field`): the UV image
with the tangential-flow quiver, the smoothed `‖v_tangential‖`, the signed `v_normal`
(where the constriction band is clearest), curl, and divergence.

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
from pathlib import Path

from seamless import FlowEstimator, KinematicsAnalyzer
from seamless.config import TwoDNeuralConfig, ThreeDNativeConfig
from seamless.vis.flow_dashboards import plot_flow_field

## Load the timeseries and pick a frame pair

The bundled sample dataset contains 2 timepoints (t=0 and t=1).
Set `PAIR = 0` to analyze this pair.

In [ ]:
proj_file = Path('data/synthetic/ellipsoid_projection.h5')
vol_file  = Path('data/synthetic/ellipsoid.h5')

full = FlowEstimator.from_projection_h5(proj_file, source_file=vol_file)

PAIR = 0                                   # frame pair (t -> t+1) to analyze
pair_frames = full.frames[PAIR:PAIR + 2]
print(f'comparing t={PAIR} -> {PAIR + 1}, uv_res={pair_frames[0].uv_res}')

## Run the three methods

Iteration counts come from the configs (the parameter-screen winners default to 1000);
we reduce them here for a quicker demo. PIV needs no training; the neural methods train
one FlowMLP for this pair.

In [ ]:
estimator = FlowEstimator(
    pair_frames,
    two_d_config=TwoDNeuralConfig(),
    three_d_config=ThreeDNativeConfig(),
)
fields = {}
for method in ['piv', '2d_neural', '3d_native']:
    fields[method] = estimator.estimate(method)[0]
    print(f'{method:10s} -> v3d {fields[method].v3d.shape}')

## Kinematics dashboard per method

`compute_eulerian` returns the full metric set; `plot_flow_field` renders the A1-style
1×5 dashboard (smoothed fields + tangential quiver).

_Note: the 3D-native dashboard runs an autograd Jacobian+Hessian over the full UV grid;
at uv_res=512 on CPU that cell can take a few minutes._

In [ ]:
analyzer = KinematicsAnalyzer(pair_frames, list(fields.values()))
for method, ff in fields.items():
    eul = analyzer.compute_eulerian(ff)
    plot_flow_field(
        ff.frame_t.max_projection, ff.frame_t.xyz_map_voxel, eul,
        title=f'{method}  (t={PAIR} -> {PAIR + 1})',
    )
    plt.show()

## Summary

All three methods produce a 3D velocity field (`FlowField.v3d`) — PIV and 2D-neural are
lifted via the NuvoMLP, 3D-native is trained on the volume. `KinematicsAnalyzer` then
gives a consistent Eulerian metric set, and `plot_flow_field` visualizes the tangential
flow and the signed normal velocity (the constriction band).